In [1]:
import sys
import os

sys.path.insert(0, os.path.abspath("../utils"))

from pathlib import Path
import re
import torch as th
from imitation.data import rollout
from imitation.data.types import Trajectory
from imitation.data import rollout
import numpy as np
from imitation.algorithms import bc
import gymnasium as gym
from imitation.data.types import Transitions
from stable_baselines3.common.torch_layers import BaseFeaturesExtractor
from stable_baselines3.common.policies import ActorCriticPolicy
import torch.nn as nn

from resident_requiem import TransferLearningLSTM, TransferLearningEfficientNetLSTM, TemporalAttentionLSTM
import gc
from IPython import get_ipython
import cv2

from utils import get_last_index

gc.collect()
th.cuda.empty_cache()

rng = np.random.default_rng()

demo_path = 'demos/'

train_path = 'imitation/'
sub_train_path = train_path + "steps/"

os.makedirs(train_path, exist_ok=True)
os.makedirs(sub_train_path, exist_ok=True)


ENT_WEIGHT= 0 # 1e-4 # 0.001 # 0.01 # 0 # 1e-3 # 0 # 1e-4
BATCH_SIZE= 384 # 512 # 128 # 128 # 256 # 64 # 256 # 128 # 64 # 128 # 32 # 64 # 128
MINI_BATCH= 128 # 64 # 32 # None # 128 # None # 64
NUMBER_OF_EPOCH= 30 # 60 # 75 # 15 #45 # 20 # 45 # 35 # 30 # 27
EPOCH_PER_FILE= 2 # 2 # 1 #3 # 1 # 2 # 1 # 3 # 2
L2= 1e-5 # 1e-4 # 1e-3 # 1e-4 # 0 # 1e-4 # 1e-5
LEARNING_RATE= 1e-4 # 4e-4  # 1e-4 # 5e-5 # 1e-4 # 2e-4
BUFFER_SIZE = 4 # 3 # 10 # 5 # 5 # 10 # 7 # 4 # 7
BUFFER_SIZE_ORIG = BUFFER_SIZE
IS_HG_DAGGER=True

if IS_HG_DAGGER:
    NUMBER_OF_EPOCH = 5

action_space = gym.spaces.MultiBinary(18)

observation_space = gym.spaces.Box(
    low=0,
    high=255,
    shape=(4, 128, 128), # 4 frames 128x128
    dtype=np.uint8,
)


last_index = get_last_index(demo_path, "demos", "pt")

print("last_index: " + str(last_index))

files_index = np.arange(last_index + 1)
# files_index = np.arange(10, last_index + 1)

class CustomActorCriticPolicy(ActorCriticPolicy):
    def __init__(self, *args, **kwargs):
        super().__init__(
            *args,
            **kwargs,
            # features_extractor_class=Re4CNN,
            # features_extractor_class=TransferLearningLSTM,
            # features_extractor_class=TransferLearningEfficientNetLSTM,
            features_extractor_class=TemporalAttentionLSTM,
            features_extractor_kwargs=dict(features_dim=256),
            # features_extractor_kwargs=dict(features_dim=512),
            # net_arch=dict(pi=[256, 256], vf=[256, 256])
            # net_arch=dict(pi=[512, 512, 256], vf=[512, 512, 256])
            # net_arch=dict(pi=[256, 256], vf=[512, 512, 256])
            # net_arch=dict(pi=[512, 256, 128], vf=[512, 256, 128]),
            net_arch=dict(pi=[256, 256], vf=[256, 256])
        )

        self.retain_graph = True
        self.step_counter = 0


policy = CustomActorCriticPolicy(
    observation_space=observation_space,
    action_space=action_space,
    lr_schedule=lambda _: th.finfo(th.float32).max,  # BC control the learning rate
)


th.serialization.add_safe_globals([Trajectory])

# TODO comment to load default/ or not using dagger
last_index_imitation = get_last_index(str(sub_train_path), "bc_policy", "zip")

if IS_HG_DAGGER:
    policy = ActorCriticPolicy.load(f"./imitation/steps/bc_policy{last_index_imitation}.zip")


bc_trainer = bc.BC(
    observation_space=observation_space,
    action_space=action_space,
    rng=rng,
    ent_weight=ENT_WEIGHT,
    batch_size=BATCH_SIZE,
    policy=policy,
    optimizer_kwargs=dict(lr=LEARNING_RATE),
    l2_weight=L2,
    minibatch_size=MINI_BATCH,
)

def concat_transitions(list_of_transitions):
    return Transitions(
        obs=np.concatenate([t.obs for t in list_of_transitions]),
        acts=np.concatenate([t.acts for t in list_of_transitions]),
        next_obs=np.concatenate([t.next_obs for t in list_of_transitions]),
        dones=np.concatenate([t.dones for t in list_of_transitions]),
        infos=np.concatenate([t.infos for t in list_of_transitions]),
    )

def fix_action_format(acts):
    """
    Fix action shape
    """
    if isinstance(acts, np.ndarray):
        if acts.ndim == 3 and acts.shape[1] == 1:
            acts = acts.squeeze(1)
        
        # if acts.dtype == np.float32 or acts.dtype == np.float64:
        #     acts = np.round(acts).astype(np.int8)

        acts = acts.astype(np.float32)
    
    return acts


# TODO set zero
epoch_count = 0 # 72 # 0

def augment_brightness(obs):
    factor = np.random.uniform(0.8, 1.2)

    obs = obs.astype(np.float32) * factor
    obs = np.clip(obs, 0, 255)

    return obs.astype(np.uint8)

def augment_noise(obs):

    noise = np.random.normal(0, 5, obs.shape)

    obs = obs.astype(np.float32) + noise
    obs = np.clip(obs, 0, 255)

    return obs.astype(np.uint8)

def random_crop(obs, crop=8):

    T, C, H, W = obs.shape

    y = np.random.randint(0, crop)
    x = np.random.randint(0, crop)

    cropped = obs[:, :, y:H-(crop-y), x:W-(crop-x)]

    resized = np.zeros((T, C, H, W), dtype=obs.dtype)

    for t in range(T):
        for c in range(C):
            resized[t,c] = cv2.resize(cropped[t,c], (W,H))

    return resized

def augmentation(obs):
    aug_obs = obs.copy()

    choice = np.random.rand()

    if choice < 0.3:
        aug_obs = augment_brightness(aug_obs)
    
    elif choice < 0.6:
        aug_obs = random_crop(aug_obs)
    
    else:
        aug_obs = augment_noise(aug_obs)

    return aug_obs

def drq_augment(obs, pad=4):
    """
    obs: (T, C, H, W)
    """

    obs_copy = obs.copy()

    T, C, H, W = obs_copy.shape

    # padding
    padded = np.pad(
        obs_copy,
        ((0,0), (0,0), (pad,pad), (pad,pad)),
        mode='edge'
    )

    # random pad
    y = np.random.randint(0, pad * 2)
    x = np.random.randint(0, pad * 2)

    cropped = padded[:, :, y:y+H, x:x+W]

    brightness = augment_brightness(cropped)

    return brightness

def random_obs_segment(obs, acts):
    size_obs = obs.shape[0]

    size_seg = int(size_obs/2)
    #double_seg = size_seg*2

    # size_seg = int(size_obs/3)
    # double_seg = size_seg*2

    # segment = np.random.randint(0, 3)
    segment = np.random.randint(0, 2)

    if segment == 0:
        obs = obs[:size_seg+1]
        acts = acts[:size_seg]
        
        return obs, acts
        
    # elif segment == 1:
    #     obs = obs[size_seg:double_seg+1]
    #     acts = acts[size_seg:double_seg]

    #     return obs, acts

    elif segment == 1:
    # elif segment == 2:
        # obs = obs[double_seg:]
        # acts = acts[double_seg:len(obs)-1]

        obs = obs[size_seg:size_seg*2]
        acts = acts[size_seg:size_seg*2-1]

        return obs, acts
    else:
        return obs, acts

def smooth_run(acts, run_idx=9, window=10):
    smoothed_acts = acts.copy()
    for t in range(len(smoothed_acts) - window):
        if smoothed_acts[t, run_idx] == 1:
            smoothed_acts[t:t+window, run_idx] = 1
    return smoothed_acts

def fix_trajectories(trajectories, aug_prob=0.5):
    fixed_trajectories = []

    for traj in trajectories:
        obs = np.array(traj.obs)

        acts = fix_action_format(np.array(traj.acts, dtype=np.int8))

        # fix run press
        acts = smooth_run(acts, run_idx=9)

        # Case (T, 1, H, W, C)
        if obs.ndim == 5 and obs.shape[1] == 1:
            obs = obs[:, 0]  # remove dimension 1 → (T, H, W, C)
        
        # Case HWC → CHW
        if obs.ndim == 4 and obs.shape[-1] == 4:
            obs = obs.transpose(0, 3, 1, 2)  # (T, 4, 64, 64)

        # print("obs shape", obs.shape)

        if obs.shape[0]> 230:
            print("obs shape:", obs.shape)

            # obs, acts = random_obs_segment(obs, acts)

            fixed_trajectories.append(
                Trajectory(
                    obs=obs,
                    acts=acts,
                    infos=traj.infos,
                    terminal=traj.terminal
                )
            )

            if np.random.random() < aug_prob:
                aug_type = np.random.choice(['brightness', 'crop', 'noise'])
                
                if aug_type == 'brightness':
                    aug_obs = augment_brightness(obs)
                    fixed_trajectories.append(Trajectory(obs=aug_obs, acts=acts, infos=traj.infos, terminal=traj.terminal))
                
                elif aug_type == 'crop':
                    aug_obs = random_crop(obs)
                    fixed_trajectories.append(Trajectory(obs=aug_obs, acts=acts, infos=traj.infos, terminal=traj.terminal))
                
                elif aug_type == 'noise':
                    aug_obs = augment_noise(obs)
                    fixed_trajectories.append(Trajectory(obs=aug_obs, acts=acts, infos=traj.infos, terminal=traj.terminal))
        
            
    return fixed_trajectories



for e in range(NUMBER_OF_EPOCH):
    np.random.shuffle(files_index)
    
    BUFFER_SIZE = BUFFER_SIZE_ORIG

    if IS_HG_DAGGER:
        epoch_count = last_index_imitation + 1
    else:
        epoch_count += 1

    if IS_HG_DAGGER:
        print(f"\n--------------- Dagger pass: {e} ------------------\n")
    else:
        print(f"\n--------------- Epoch: {epoch_count} ------------------\n")
    print("files_index: ", files_index)
    
    buffer = []
    buffer_files = []
    
    for idx, file_idx in enumerate(files_index):
        trajectories = th.load(demo_path + f"demos{file_idx}.pt", weights_only=False)
        fixed_trajectories = fix_trajectories(trajectories)
        np.random.shuffle(fixed_trajectories)
        transitions = rollout.flatten_trajectories(fixed_trajectories)
        
        buffer.append(transitions)
        buffer_files.append(file_idx)
        
        del transitions, fixed_trajectories, trajectories
        

        if len(buffer) == BUFFER_SIZE or (idx == len(files_index) - 1 and len(buffer) > 0):
            if idx == len(files_index) - 1 and len(buffer) < BUFFER_SIZE:
                print(f"Last batch {len(buffer)} files (less than BUFFER_SIZE)")
            
            merged = concat_transitions(buffer)
            print(f"Processing files: {buffer_files}")
            
            bc_trainer.set_demonstrations(merged)
            del merged
            
            bc_trainer.train(
                n_epochs=EPOCH_PER_FILE,
                progress_bar=True,
                log_interval=200
            )
            
            bc_trainer.policy.save(sub_train_path + f"bc_policy{epoch_count}.zip")
            
            buffer.clear()
            buffer_files.clear()
            th.cuda.empty_cache()
            gc.collect()


# bc_trainer.policy.save(train_path + f"bc_policy{last_index_imitation + 1}.zip")

gc.collect()
bc_trainer._demonstrations = None
bc_trainer._demonstrations_tensor = None
del bc_trainer

th.cuda.empty_cache()

print("Force cell kernel reset")
get_ipython().kernel.do_shutdown(restart=True)

last_index: 19


D:\Python311\Lib\site-packages\stable_baselines3\common\policies.py:176: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  saved_variables = th.load(path, map_location=device)



--------------- Dagger pass: 0 ------------------

files_index:  [ 9 17 16  4  3  2 19 14  0  7  6 10 12 18  5 13 11 15  1  8]
obs shape: (2668, 4, 128, 128)
obs shape: (2339, 4, 128, 128)
obs shape: (2374, 4, 128, 128)
obs shape: (1750, 4, 128, 128)
obs shape: (2776, 4, 128, 128)
obs shape: (2245, 4, 128, 128)
obs shape: (1664, 4, 128, 128)
obs shape: (2816, 4, 128, 128)
obs shape: (2589, 4, 128, 128)
obs shape: (1850, 4, 128, 128)
obs shape: (2930, 4, 128, 128)
obs shape: (2968, 4, 128, 128)
obs shape: (2647, 4, 128, 128)
obs shape: (2650, 4, 128, 128)
obs shape: (2552, 4, 128, 128)
obs shape: (2826, 4, 128, 128)
obs shape: (2855, 4, 128, 128)
obs shape: (3205, 4, 128, 128)
obs shape: (2481, 4, 128, 128)
obs shape: (2653, 4, 128, 128)
obs shape: (2789, 4, 128, 128)
obs shape: (349, 4, 128, 128)
obs shape: (2643, 4, 128, 128)
obs shape: (2775, 4, 128, 128)
obs shape: (2998, 4, 128, 128)
obs shape: (3039, 4, 128, 128)
obs shape: (2534, 4, 128, 128)
obs shape: (2718, 4, 128, 128)
obs s

2batch [00:03,  1.62s/batch]

--------------------------------
| batch_size        | 128      |
| bc/               |          |
|    batch          | 0        |
|    ent_loss       | 0        |
|    entropy        | 1.37     |
|    epoch          | 0        |
|    l2_loss        | 0.0587   |
|    l2_norm        | 5.87e+03 |
|    loss           | 2.04     |
|    neglogp        | 1.98     |
|    prob_true_act  | 0.479    |
|    samples_so_far | 384      |
--------------------------------


...

1801batch [02:13, 13.26batch/s]

--------------------------------
| batch_size        | 128      |
| bc/               |          |
|    batch          | 600      |
|    ent_loss       | 0        |
|    entropy        | 1.58     |
|    epoch          | 1        |
|    l2_loss        | 0.0603   |
|    l2_norm        | 6.03e+03 |
|    loss           | 1.66     |
|    neglogp        | 1.6      |
|    prob_true_act  | 0.485    |
|    samples_so_far | 230784   |
--------------------------------


2003batch [02:28, 13.41batch/s]
2004batch [02:28, 13.47batch/s]


Force cell kernel reset


{'status': 'ok', 'restart': True}